In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("p2.ipynb")

# Project 2 – Loan Applications: Borrower Profiles and Tax Burdens
## DATA 2201, Fall 2026

## Instructions

---

### Working on the Project

    - When you submit the project, however, we will use hidden tests to check your answers more thoroughly.
 
- **Do not change the function names!** The functions are how your assignment is graded, and they are graded by their name.
- You are encouraged to write your own additional helper functions to solve the project, as long as they also end up in `p2.ipynb`.

### Warning!

Many questions in the project intentionally build off of each other and the final result matters! In fact, you can "get a question correct," but only receive partial credit for it because a previous answer was wrong.

For any questions that related with number rounding, please be aware that `np.round()` and `round()` function differently. For this assignment, use `round()` when needed as it is the function we will utilize in the tests.



In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

### Helper Function for Kernel Density Estimation (KDE) Plots

In [ ]:
def create_kde_plotly(df, group_col, group1, group2, vals_col, title=''):
    # Equivalent density demonstration without figure_factory.create_distplot.
    fig, ax = plt.subplots()
    for group in [group1, group2]:
        values = df.loc[df[group_col].eq(group), vals_col].dropna()
        if len(values) > 1 and values.nunique() > 1:
            sns.kdeplot(x=values, label=str(group), ax=ax)
    ax.set(title=title.replace('<br>', ' '), xlabel=vals_col, ylabel='Density')
    ax.legend(title=group_col)
    plt.show()
    return ax

## About the Assignment

[LendingClub](https://www.lendingclub.com/) is a platform that allows individuals to borrow money – that is, take on **loans**. They've made available a massive dataset with information on millions of loans they've processed. The entire dataset is over 300 MB large, and so we won't work with it here – instead, we'll work with a sample from it.

Run the cell below to load in this sample from the file `data/loans.csv`.

In [ ]:
loans_path = Path('data') / 'loans.csv'
loans = pd.read_csv(loans_path)
loans.head()

Each row of the dataset corresponds to a different loan that the LendingClub approved and paid out. Some of the key columns are:
- `'loan_amnt' (float)`: the amount of the loan, or how much the borrower borrowed.
- `'issue_d' (str)`: the date on which the loan was issued.
- `'term' (str)`: the length of the loan, that is, the amount of time the borrower has to pay the loan back.
- `'int_rate' (float)`: the interest rate the borrower will pay on their loan amount.

First, it's worth exploring the distribution of loan amounts. You'll see that the largest possible loan given out through LendingClub was \\$40,000.

In [ ]:
loans['loan_amnt'].describe()

Note that when a borrower is approved for a loan, they are presented with multiple offers with different loan terms and interest rates. **Larger interest rates make the loan more expensive for the borrower – as a borrower, you want a lower interest rate!** You'll note that even for the same loan amount, different borrowers were approved for different terms and interest rates. Take a look below:

In [ ]:
loans.loc[loans['loan_amnt'] == 3600, ['loan_amnt', 'term', 'int_rate']]

So, why do different borrowers receive different terms and interest rates, despite asking for the same amount of money? The factors that influence loan offers are complex, but it's [known](https://www.bankofamerica.com/smallbusiness/resources/post/factors-that-impact-loan-decisions-and-how-to-increase-your-approval-odds/) that some of the key factors are employment status, annual income, and credit score, among other things.

In this project, we will **explore how various borrower characteristics are related to one another**, in an attempt to better understand the complexity behind loans. It's important to remember, though, that this dataset only contains information about actually approved loans, **not** all loan applications.

---

<a id="outline"></a>

### Navigating the Project

Click the links below to navigate to each section.

- [Part 1: Understanding Lender Decision-Making](#part1)
    - [Question 1](#Question-1)
    - [Question 2](#Question-2)
    - [Question 3](#Question-3)
    - [Question 4](#Question-4)
- [Part 2: Calculating Disposable Incomes](#part2)
    - [Question 5](#Question-5)
    - [Question 6](#Question-6)
    - [Question 7](#Question-7)
    - [Question 8](#Question-8)
    - [Question 9](#Question-9)
    - [Question 10](#Question-10)
    - [Question 11](#Question-11)

<a id="part1"></a>

---
## Part 1: Understanding Lender Decision-Making

([return to the outline](#outline))

As mentioned above, the dataset we have only has information about approved loans – we don't have information about individuals who applied for loans and weren't approved. That means that we can't directly study what distinguishes successful loan applications from unsuccessful ones.

We'll start by understanding the **quantitative risks** that lenders assess when deciding what terms and interest rates to give to borrowers. Lenders typically charge higher interest rates to borrowers they perceive as high-risk. This practice serves several purposes: it offsets potential losses from unpaid debts, discourages excessively risky lending, and ensures that the higher returns from these loans can cover defaults. 

The first quantitative metric we'll look at is **debt-to-income (DTI) ratio**. Understanding the impact of such a metric on interest rates helps us in evaluating how lenders quantify risk.

<!-- The realm of lender decision-making is complex and multifaceted, influenced by a blend of traditional and non-traditional metrics. Our motivation is to delve into this domain and comprehend financial indicators, starting with: 
 -->
 
<!-- **Quantitative Risks:** Traditional metrics like **Debt-to-Income (DTI)** ratio and annual income are quantifiable and offer a concrete basis for assessing a borrower's financial health. 
 -->
<br/>

$$
\text{DTI} = \left( \frac{\text{Total Monthly Debt Payments}}{\text{Gross Monthly Income}} \right) \cdot 100 
$$

<br/>

Note that you don't need to calculate DTIs – they are already provided for us in the `'dti'` column of `loans`.


A low DTI indicates that a borrower is less likely to face financial strain from taking on additional debt, making them a lower risk to lenders. Conversely, a high DTI may signal financial overextension, suggesting a higher risk for default – that is, a higher risk that the borrower won't pay back the loan. 

When a lender perceives a higher risk of default, they often charge a higher interest rate. Let's see if this correlation is present in our dataset by looking a scatter plot of interest rate (`'int_rate'`) vs DTI (`'dti'`).

In [ ]:
sample_set = loans.sample(200, random_state=1)

fig = px.scatter(sample_set, x='dti', y='int_rate', trendline='ols',
                 labels={'dti': 'Debt-to-Income Ratio', 
                         'int_rate': 'Interest Rate (%)'},
                 trendline_color_override='orange',
                 title='Interest Rate vs. Debt-to-Income Ratio')

fig.show()

In [ ]:
# The same plot can be created in "seaborn"
sns.regplot(data=sample_set, x='dti', y='int_rate')
plt.title("Interest Rate vs. Debt-to-Income Ratio")
plt.xlabel("Debt-to-Income Ratio")
plt.ylabel("Interest Rate (%)");

### Provided preparation (ungraded)
The original cleaning and tax setup are provided so that the new questions do not repeat those tasks. The historical state rates retain the original exercise's rounding to two decimal places. Use `loans` for borrower analyses and `state_taxes` for tax analyses. No deductions, credits, filing-status differences, or local taxes are modeled.

In [ ]:
def prepare_example_loans(df):
    result = df.copy(deep=True)
    result['issue_d'] = pd.to_datetime(result['issue_d'], format='%b-%Y')
    result['term'] = result['term'].astype(str).str.strip().str.split().str[0].astype(int)
    result['emp_title'] = result['emp_title'].str.lower().str.strip().replace({'rn': 'registered nurse'})
    result['term_end'] = result.apply(lambda row: row['issue_d'] + pd.DateOffset(months=int(row['term'])), axis=1)
    return result

def prepare_example_taxes(df):
    result = df.dropna(how='all').copy()
    result['State'] = result['State'].mask(result['State'].astype(str).str.contains('(', regex=False)).ffill()
    rate = result['Rate'].astype(str).str.strip().str.lower().replace({'none': '0'})
    result['Rate'] = (pd.to_numeric(rate.str.replace('%', '', regex=False)) / 100).round(2)
    lower = result['Lower Limit'].fillna('0').astype(str).str.strip().replace({'none': '0'})
    result['Lower Limit'] = pd.to_numeric(lower.str.replace('$', '', regex=False).str.replace(',', '', regex=False)).astype(int)
    return result

FEDERAL_BRACKETS = [(0.1, 0), (0.12, 11000), (0.22, 44725), (0.24, 95375), (0.32, 182100), (0.35, 231251), (0.37, 578125)]
example_brackets = FEDERAL_BRACKETS.copy()

def example_tax(income, brackets):
    ordered = sorted(brackets, key=lambda pair: pair[1])
    return sum(rate * max(0.0, min(income, ordered[i+1][1] if i+1 < len(ordered) else np.inf) - lower)
               for i, (rate, lower) in enumerate(ordered))

loans = prepare_example_loans(pd.read_csv(loans_path))
state_taxes_raw = pd.read_csv(Path('data') / 'state_taxes_raw.csv')
state_taxes = prepare_example_taxes(state_taxes_raw)
with open(Path('data') / 'state_mapping.json', encoding='utf-8') as handle:
    state_mapping = json.load(handle)
loans['has_ps'] = loans['desc'].notna()
loans['fico_range_low_cat'] = pd.cut(loans['fico_range_low'], [580, 670, 740, 800, 850], right=False)


### Original demonstrations (ungraded)
Retain the original 36/60-month, credit-score, personal-statement, and $75,000 tax examples as context. Score intervals in this exercise are left-inclusive and right-exclusive; 850 is outside these displayed bins.

In [ ]:
display(loans['issue_d'].dt.year.value_counts().sort_index())
px.scatter(loans, x='fico_range_low', y='int_rate', labels={'fico_range_low': 'Credit Score', 'int_rate': 'Interest Rate (%)'}, title='Interest Rate vs. Credit Score').show()
sns.catplot(data=loans, x='fico_range_low_cat', y='int_rate', hue='term', kind='box', height=4, aspect=1.4, palette=['#9370db', '#FECF49'])
plt.xlabel('Credit Score Range')
plt.ylabel('Interest Rate (%)')
plt.title('Interest Rate vs. Credit Score')
plt.show()
display(loans.groupby('has_ps')['int_rate'].describe())
create_kde_plotly(loans, 'has_ps', True, False, 'int_rate', title='Distributions of Interest Rates Based on Inclusion of Personal Statement')
example_owed = example_tax(75000, example_brackets)
print('Tax on $75,000 in the original classroom example:', round(example_owed, 2))


---

<a id="Question-1"></a>

### Question 1: Borrower Burden Features (8 points, autograded)

[Return to the outline](#outline)

Implement `burden_features(df)`. Return a copy with three added columns:

- `loan_to_income = loan_amnt / annual_inc`
- `monthly_income = annual_inc / 12`
- `estimated_monthly_debt = monthly_income * dti / 100`

For zero, negative, or missing annual income, all three added values must be NaN. Missing loan amounts or DTI propagate only to the corresponding calculation. Keep all original columns, rows, and index values. Use vectorized operations.

In [ ]:
def burden_features(df):
    # Copy the input DataFrame.
    out = ...
    # Add the three required features.
    ...
    return out

In [ ]:
grader.check("q1")

---

<a id="Question-2"></a>

### Question 2: Loan Summaries by Grade (10 points, autograded)

[Return to the outline](#outline)

Implement `grade_summary(df)` to return a DataFrame indexed by `grade`, sorted alphabetically, with two columns in this order:

- `n_loans`: total number of loans in each grade, including those with missing interest rates.
- `median_rate`: median interest rate (`int_rate`) for each grade.

Exclude rows with missing grades. Ignore missing interest rates when calculating medians. Do not modify the input DataFrame.

In [ ]:
def grade_summary(df):
    # Calculate loan counts and median interest rates by grade.
    summary = ...
    return summary

In [ ]:
grader.check("q2")

---

<a id="Question-3"></a>

### Question 3: Median Interest Rate by Credit Band (15 points, autograded)

[Return to the outline](#outline)

Implement `credit_summary(df)` to calculate the median interest rate for
each credit band.

Use `pd.cut()` to group `fico_range_low` into:

- `Fair`: 580 ≤ score < 670
- `Good`: 670 ≤ score < 740
- `Very Good`: 740 ≤ score < 800
- `Excellent`: 800 ≤ score < 850

Store the median `int_rate` for each band in a Series named `median_rates`
and return it. Keep the bands in the order listed above.

Exclude missing scores and scores outside these ranges. Ignore missing
interest rates when calculating medians. Empty bands should have `NaN`.
Do not modify the input DataFrame.

**Hint:** Use `right=False` in `pd.cut()` and `observed=False` in `groupby()`.

In [ ]:
def credit_summary(df):
    credit_data = df.copy()

    # Create the four credit bands.
    credit_data['credit_band'] = pd.cut(
        credit_data['fico_range_low'],
        bins=...,
        labels=...,
        right=False
    )

    # Calculate the median interest rate for each band.
    median_rates = ...

    return median_rates

In [ ]:
grader.check("q3")

---

<a id="Question-4"></a>

### Question 4: Two-Sided Median Permutation Test (20 points, autograded)

[Return to the outline](#outline)

Do loans with and without a statement have different median interest rates?

Implement `median_statement_test(df, N=1000, seed=42)` using the absolute
difference between the two groups' median interest rates.

Exclude rows with missing `int_rate`. Identify statement presence using
`desc.notna()`.

Return a dictionary with:

- `observed`: the observed absolute difference in medians.
- `simulated`: a NumPy array of `N` differences obtained by shuffling
  statement-presence labels.
- `p_value`: `(1 + count(simulated >= observed)) / (N + 1)`.

Use `np.random.default_rng(seed)` and exactly one `rng.permutation(labels)`
per repetition. Keep interest rates in their original order.

Raise `ValueError` if `N` is not a positive integer (excluding booleans)
or either group is absent after filtering. Do not modify `df`.

In [ ]:
def median_statement_test(df, N=1000, seed=42):
    # Check that N is a positive integer, excluding booleans.
    ...

    # Remove missing rates and identify statement presence.
    data = ...
    rates = ...
    labels = ...

    # Raise ValueError if either group is absent.
    ...

    # Calculate the absolute difference in group medians.
    observed = ...

    rng = np.random.default_rng(seed)
    simulated = np.zeros(N)

    for i in range(N):
        shuffled = rng.permutation(labels)
        simulated[i] = ...

    # Calculate the p-value, including ties.
    p_value = ...

    result = dict(observed=observed, simulated=simulated, p_value=p_value)
    return result

In [ ]:
grader.check("q4")

<a id="part2"></a>

---
## Part 2: Tax allocation and affordability

[Return to the outline](#outline)

Use the supplied historical tax helpers and original state files. Bracket lists in Questions 5 and 8 are nonempty, have unique nonnegative lower limits starting at zero, and have nonnegative rates. Incomes are finite and nonnegative. These questions model the same simplified tax system as the original examples.

---

<a id="Question-5"></a>

### Question 5: Tax Allocation by Bracket (12 points, autograded)

[Return to the outline](#outline)

For an income of $25,000, the first $10,000 is taxed at 10%,
the next $10,000 at 20%, and the remaining $5,000 at 30%.

Create a DataFrame named `tax_allocation` with three columns:

- `rate`: the tax rates, expressed as decimals.
- `taxable_amount`: the income taxed in each bracket.
- `tax`: the taxable amount multiplied by the tax rate.

List the brackets from the lowest rate to the highest.
Do not round your results.

In [ ]:
tax_allocation = pd.DataFrame({
    "rate": ...,
    "taxable_amount": ...
})
tax_allocation["tax"] = ...
tax_allocation

In [ ]:
grader.check("q5")

---

<a id="Question-6"></a>

### Question 6: State Bracket Intervals (13 points, autograded)

[Return to the outline](#outline)

Implement `state_intervals(cleaned)` using the provided cleaned
three-column state-tax table.

Return a DataFrame with these columns in order:

- `State`
- `Rate`
- `Lower Limit`
- `Upper Limit`

**Requirements**

- Sort by `State`, then by `Lower Limit`.
- Reset the index to start at 0.
- Set each upper limit to the next lower limit **within the same state**.
- Use `np.inf` for each state's last bracket.
- Keep the original rounded rates.
- Return a DataFrame, not bracket lists.

Assume lower limits are valid and distinct within each state.

In [ ]:
def state_intervals(cleaned):
    out = ...
    out["Upper Limit"] = ...
    return out

In [ ]:
grader.check("q6")

---

<a id="Question-7"></a>

### Question 7: State Coverage Audit (13 points, autograded)

[Return to the outline](#outline)

Implement `state_coverage(df, cleaned, mapping)` to check which loan
states have a tax schedule.

The dictionary `mapping` converts state names in `cleaned["State"]`
to two-letter state codes.

Return a DataFrame with one row for each distinct, nonmissing
`addr_state` in `df`, containing:

- `State`: the two-letter state code.
- `n_loans`: the number of loans in that state.
- `has_tax_schedule`: `True` if the code appears among the mapped
  states in `cleaned`; otherwise, `False`.

Sort by `State` and reset the index to start at 0.
Keep unknown state codes with `has_tax_schedule = False`.
Do not merge tax lists into the loan table.

In [ ]:
def state_coverage(df, cleaned, mapping):
    counts = ...
    out = ...
    known = ...
    out["has_tax_schedule"] = ...
    return out

In [ ]:
# Run this cell before the tests. Do not modify the code below.
test_loans = pd.DataFrame({"addr_state": ["MI", "CA", "MI"]})
taxes = pd.DataFrame({"State": ["Michigan"]})
mapping = {"Michigan": "MI"}
result = state_coverage(test_loans, taxes, mapping)

In [ ]:
grader.check("q7")

---

<a id="Question-8"></a>

### Question 8: How Much of a Raise Do You Keep? (12 points, autograded)

[Return to the outline](#outline)

An employee earns **\$48,000 per year** and receives a **\$6,000 annual raise**, bringing their salary to **\$54,000**.

For this exercise, use the following **simplified income-tax schedule**:

| Portion of annual income | Tax rate |
|--------------------------|----------|
| First \$15,000 | 10% |
| Above \$15,000 through \$50,000 | 20% |
| Above \$50,000 | 30% |

Each rate applies only to the portion of income within that bracket.

Assume the full salary is taxable and ignore deductions and other taxes.

Calculate:

- `additional_tax`: the extra annual tax resulting from the raise.
- `retained_increase`: the annual raise remaining after the additional tax.
- `incremental_rate`: the fraction of the raise paid in additional tax.

Store the results in a Series named `raise_impact`, with these index
labels in order: `additional_tax`, `retained_increase`, `incremental_rate`.

Express the rate as a decimal. Do not round.

In [ ]:
additional_tax = ...
retained_increase = ...
incremental_rate = ...

raise_impact = pd.Series(
    ...,
    index=["additional_tax", "retained_increase", "incremental_rate"]
)
raise_impact

In [ ]:
grader.check("q8")

---

<a id="Question-9"></a>

### Question 9: Monthly Affordability (10 points, autograded)

[Return to the outline](#outline)

Implement `monthly_affordability(df)` to examine how loan payments
affect a borrower's monthly income.

The DataFrame contains:

- `disposable_income`: annual income after taxes.
- `loan_amnt`: total loan amount.
- `term`: loan duration in months.

Return a copy with three new columns:

- `monthly_income`: annual disposable income divided by 12.
- `monthly_payment`: loan amount divided by the term.
- `remaining_income`: monthly income minus the monthly payment.

Assume all values are present and positive. Ignore loan interest.
Do not modify the original DataFrame.

In [ ]:
def monthly_affordability(df):
    out = df.copy()
    out["monthly_income"] = ...
    out["monthly_payment"] = ...
    out["remaining_income"] = ...
    return out

In [ ]:
# Run this cell before the tests. Do not modify the code below.
sample = pd.DataFrame({
    "disposable_income": [48000, 36000],
    "loan_amnt": [12000, 6000],
    "term": [24, 12]
})
result = monthly_affordability(sample)

In [ ]:
grader.check("q9")

<!-- BEGIN QUESTION -->

---

<a id="Question-10"></a>

### Question 10: Loan Amounts by State (12 points, manually graded)

[Return to the outline](#outline)

Using the `loans` DataFrame, create a **horizontal bar chart** showing
the **10 states with the highest median loan amounts**.

- Calculate the median `loan_amnt` for each `addr_state`.
- Select the 10 states with the highest medians.
- Place the state with the highest median at the top.
- Include a descriptive title and label both axes.
- Show loan amounts in USD.

**Rubric (12 points):** Correct medians (4 points), correct top 10 and ordering
(3 points), accurate horizontal bar chart (3 points), title and axis
labels (2 points).

In [ ]:
median_loans = ...
top10 = ...

...
plt.tight_layout()
plt.show()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

---

<a id="Question-11"></a>

### Question 11: DTI Distributions (13 points, manually graded)

[Return to the outline](#outline)

Using `loans`, create a **violin plot** comparing debt-to-income ratios
(`dti`) by personal-statement presence and loan term.

- Include only 36-month and 60-month loans.
- Exclude rows with missing `dti`.
- Use `desc.notna()` to label statements as `"Present"` or `"Missing"`.
- Place statement presence on the horizontal axis and DTI on the vertical axis.
- Show the two loan terms side by side within each statement group,
  using different colors.
- Include a descriptive title, axis labels, and a legend.

**Rubric (13 points):** Data preparation (4 points), correct violin plots and grouping
(5 points), title, axis labels, and legend (4 points).

In [ ]:
plot_data = ...
plot_data["statement"] = ...

fig, ax = plt.subplots(figsize=(8, 5))
...
plt.tight_layout()
plt.show()

<!-- END QUESTION -->


## 🎉 Congratulations — You’ve Finished Project 2!  
You’ve now completed all parts of **Project 2**.  

Before wrapping up:  
- ✅ Review your notebook to ensure **all code runs smoothly** from top to bottom.  
- 📤 Submit your completed notebook and responses **by the project deadline** on Gradescope.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)